# Are faster replies shorter? (week 5, relationships)

Builds the Analysis 16 chart from `analysis-log.md`.

**Decisions, made with the student along the way (details in the log):**

- Reply = a message directly after someone else's message, gap < 401 min (the conversation cut from Analysis 15).
- Gap in minutes with Analysis 15's corrected jitter, then **floored at 1 minute** ("≤ 1 min") for plot and fit:
  on a log axis the jitter's made-up sub-minute positions would pull the slope.
- Length = words, an emoji counts as 1 word (the stored `n_words` uses a different rule).
- Media, deleted messages, locations and contact cards are dropped; the "edited" marker is stripped.
- Both axes log, so a straight line is a ratio claim. Lowess is a check on the straight line, not a claim.
- The claim is about 9 people: one slope per author, uncertainty from resampling **days**.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess

from wa_analyzer.data import load_own_chat

CUT = 401  # minutes -- conversation cut from Analysis 15
FLOOR = 1  # minutes -- every faster reply counts as "<= 1 min"
SEED = 42
NOT_A_TEXT = re.compile(r"<Media weggelaten>|bericht is verwijderd|locatie gedeeld|^locatie: |\.vcf \(bestand")
EMOJI = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF]")
WORD = re.compile(r"[^\W_]+(?:['’][^\W_]+)*")

own = load_own_chat()
# same fix as 06-11: `timestamp` is stored as strings; the `+00:00` is only a label
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)
own = own.sort_values("timestamp", kind="stable").reset_index(drop=True)
own["day"] = own["timestamp"].dt.date

In [ ]:
rng = np.random.default_rng(SEED)
recorded = own["timestamp"].diff().dt.total_seconds() / 60
s1, s2 = rng.uniform(0, 1, len(own)), rng.uniform(0, 1, len(own))
own["gap"] = np.where(recorded == 0, np.abs(s2 - s1), recorded + s2 - s1)

text = own["message"].fillna("").astype(object).str.replace("<Dit bericht is bewerkt>", "", regex=False)
own["words"] = [len(WORD.findall(t)) + len(EMOJI.findall(t)) for t in text]
not_a_text = np.array([bool(NOT_A_TEXT.search(t)) for t in text])

is_reply = (
    own["author"].ne(own["author"].shift())
    & recorded.notna()
    & (own["gap"] > 0)
    & (own["gap"] < CUT)
    & ~not_a_text
    & (own["words"] > 0)
)
replies = own[is_reply].copy()
replies["log_gap"] = np.log10(replies["gap"].clip(lower=FLOOR))
replies["log_words"] = np.log10(replies["words"])
impossible = (replies["words"] > 40 * replies["gap"]).mean()
print(f"{len(replies):,} replies, {impossible:.1%} with more words than fit in their gap at 40 words/min (kept)")

In [ ]:
def slope(frame):
    return stats.linregress(frame["log_gap"], frame["log_words"]).slope


days = np.array(sorted(replies["day"].unique()))
by_day = dict(tuple(replies.groupby("day")))
boot = []
for _ in range(1000):
    sample = pd.concat([by_day[d] for d in rng.choice(days, len(days))])
    boot.append(sample.groupby("author").apply(slope, include_groups=False))
boot = pd.DataFrame(boot)

per_author = pd.DataFrame(
    {
        "replies": replies.groupby("author").size(),
        "slope": replies.groupby("author").apply(slope, include_groups=False),
        "lo": boot.quantile(0.025),
        "hi": boot.quantile(0.975),
    }
).sort_values("slope")
pooled = stats.linregress(replies["log_gap"], replies["log_words"])
print(f"pooled slope {pooled.slope:.3f} (10x slower reply = {10**pooled.slope - 1:.0%} more words), r = {pooled.rvalue:.2f}")
per_author.round(3)

## Verification (stage 6) — rules fixed before running

| Check | Changes the claim if |
|---|---|
| T sweep: only replies with gap ≥ 1, 3, 5, 10 min | at T = 10 the pooled slope is ≈ 0 **and** most authors' intervals cross 0 |
| Null: shuffle word counts within author × day, 1,000× | the real pooled slope sits inside the shuffled range |
| Third variable: + log length of the previous message | the gap slope shrinks by more than half (growth is reported too: discussions are fast *and* long) |
| Held-out years: 2020–22 vs 2023–26 | direction flips, or fewer than half of the authors positive in 2023–26 |
| Robustness: raw jitter, fit only ≥ 1 min | reported only |

In [ ]:
def slopes_with_intervals(frame, x="log_gap", y="log_words", draws=1000):
    """Slope per author, with a 95% interval from resampling days (sums per day, so it is fast)."""
    rows = {}
    for author, part in frame.groupby("author"):
        g = part.assign(xy=part[x] * part[y], xx=part[x] ** 2).groupby("day")
        sums = pd.DataFrame({"n": g.size(), "x": g[x].sum(), "y": g[y].sum(), "xy": g["xy"].sum(), "xx": g["xx"].sum()}).to_numpy()
        weights = rng.multinomial(len(sums), np.full(len(sums), 1 / len(sums)), size=draws)
        n, sx, sy, sxy, sxx = (weights @ sums).T
        boot_slopes = (sxy - sx * sy / n) / (sxx - sx**2 / n)
        rows[author] = {
            "slope": stats.linregress(part[x], part[y]).slope,
            "lo": np.percentile(boot_slopes, 2.5),
            "hi": np.percentile(boot_slopes, 97.5),
        }
    return pd.DataFrame(rows).T


def summary(frame, label):
    per = slopes_with_intervals(frame)
    return {
        "version": label,
        "replies": len(frame),
        "pooled_slope": stats.linregress(frame["log_gap"], frame["log_words"]).slope,
        "authors_positive": int((per["slope"] > 0).sum()),
        "intervals_above_0": int((per["lo"] > 0).sum()),
        "intervals_below_0": int((per["hi"] < 0).sum()),
    }

### Typing time and parallel writing: the T sweep

In [ ]:
sweep = []
for threshold in [1, 3, 5, 10]:
    kept = replies[replies["gap"] >= threshold].assign(log_gap=lambda d: np.log10(d["gap"]))
    sweep.append(summary(kept, f"gap >= {threshold} min"))
sweep = pd.DataFrame([summary(replies, "main (floor 1 min)")] + sweep)
sweep.round(3)

### Null: no link between a reply's gap and its length

In [ ]:
group = replies["author"] + "|" + replies["day"].astype(str)
codes = pd.factorize(group)[0]
x = replies["log_gap"].to_numpy()
y = replies["log_words"].to_numpy()
null = []
for _ in range(1000):
    order = np.lexsort((rng.random(len(y)), codes))
    shuffled = np.empty_like(y)
    shuffled[np.sort(order)] = y[order]  # words permuted within each author x day
    null.append(stats.linregress(x, shuffled).slope)
null = np.array(null)
print(f"shuffled slopes: {null.min():.3f} to {null.max():.3f} (99th pct {np.percentile(null, 99):.3f}); real {pooled.slope:.3f}")
print(f"shuffles at least as steep as the real one: {(null >= pooled.slope).sum()} of {len(null)}")

### Third variable: how long was the message being replied to?

In [ ]:
previous_words = own["words"].shift().where(~pd.Series(not_a_text).shift(fill_value=True).to_numpy())
with_previous = replies.assign(log_previous=np.log10(previous_words[replies.index].clip(lower=1))).dropna(subset=["log_previous"])
print(f"{len(replies) - len(with_previous):,} replies dropped: they answer a media/system line")


def gap_slope(frame, controls):
    design = np.column_stack([np.ones(len(frame)), frame["log_gap"]] + [frame[c] for c in controls])
    return np.linalg.lstsq(design, frame["log_words"], rcond=None)[0][1]


third = pd.DataFrame(
    {
        author: {"gap_slope_alone": gap_slope(part, []), "gap_slope_with_previous": gap_slope(part, ["log_previous"])}
        for author, part in with_previous.groupby("author")
    }
).T
third.loc["pooled"] = [gap_slope(with_previous, []), gap_slope(with_previous, ["log_previous"])]
third["change"] = third["gap_slope_with_previous"] / third["gap_slope_alone"] - 1
print(f"previous length vs reply length, r = {np.corrcoef(with_previous['log_previous'], with_previous['log_words'])[0, 1]:.2f}; "
      f"previous length vs gap, r = {np.corrcoef(with_previous['log_previous'], with_previous['log_gap'])[0, 1]:.2f}")
third.round(3)

### Held-out years: fitted on 2020–22, checked on 2023–26

In [ ]:
year = replies["timestamp"].dt.year
halves = pd.DataFrame([summary(replies[year <= 2022], "2020-2022"), summary(replies[year >= 2023], "2023-2026")])
halves.round(3)

### Robustness: the jitter choices from stage 4 (reported, no rule)

In [ ]:
raw_jitter = replies.assign(log_gap=np.log10(replies["gap"].clip(lower=0.01)))
only_minute_plus = replies[replies["gap"] >= 1].assign(log_gap=lambda d: np.log10(d["gap"]))
pd.DataFrame([summary(replies, "floor 1 min (main)"), summary(raw_jitter, "raw jitter"), summary(only_minute_plus, "only >= 1 min")]).round(3)

## Final chart

In [ ]:
fig, (ax, side) = plt.subplots(1, 2, figsize=(11, 5.6), gridspec_kw={"width_ratios": [2.4, 1]})

ax.scatter(replies["log_gap"], replies["log_words"], s=6, alpha=0.06, color="grey", lw=0)
xs = np.linspace(replies["log_gap"].min(), replies["log_gap"].max(), 50)
ax.plot(xs, pooled.intercept + pooled.slope * xs, color="tab:orange", lw=2.5)
# evaluated on a grid: thousands of replies tie on the "<= 1 min" floor
smooth = lowess(replies["log_words"], replies["log_gap"], frac=0.3, xvals=xs)
ax.plot(xs, smooth, color="dimgrey", lw=1, ls="--")
ax.annotate("straight line", (xs[-1], pooled.intercept + pooled.slope * xs[-1]), xytext=(4, 6),
            textcoords="offset points", color="tab:orange", fontsize=9)
ax.annotate("lowess", (xs[-1], smooth[-1]), xytext=(4, -12),
            textcoords="offset points", color="dimgrey", fontsize=9)
gap_ticks = [1, 3, 10, 30, 100, 300]
ax.set_xticks(np.log10(gap_ticks), ["≤ 1"] + [str(t) for t in gap_ticks[1:]])
word_ticks = [1, 3, 10, 30, 100]
ax.set_yticks(np.log10(word_ticks), [str(t) for t in word_ticks])
ax.set_xlabel("minutes since someone else's message (log)")
ax.set_ylabel("words in the reply (log)")
ax.spines[["top", "right"]].set_visible(False)

y = np.arange(len(per_author))
side.hlines(y, per_author["lo"], per_author["hi"], color="grey", lw=2)
side.plot(per_author["slope"], y, "o", color="tab:orange")
side.axvline(0, color="black", lw=0.8)
side.set_yticks(y, per_author.index, fontsize=9)
side.set_xlabel("slope per author\n(95% interval, days resampled)")
side.spines[["top", "right", "left"]].set_visible(False)
side.tick_params(axis="y", length=0)

whole_range = 10 ** (pooled.slope * np.log10(300)) - 1
slow_only = 10 ** sweep.loc[sweep["version"] == "gap >= 10 min", "pooled_slope"].item() - 1
fig.suptitle(f"When people take time to respond, their reply has roughly {whole_range:.0%} more words",
             x=0.02, ha="left", fontsize=13, fontweight="bold")
fig.text(0.02, 0.905, f"A reply after about 5 hours vs one within a minute (about 5 vs 4 words). "
         f"Same direction for {(per_author['slope'] > 0).sum()} of 9 people.", fontsize=10, color="dimgrey")
footnote = [
    f"{len(replies):,} replies, 2020-2026; gap = minutes since someone else's message, under 1 min counted as 1; emoji = 1 word.",
    f"Strongest in slower replies: after 10+ min a 10x slower reply has {slow_only:.0%} more words (all 9 people).",
    f"Checks: 0 of 1,000 shuffles (within person and day) as steep; length of the message replied to changes it "
    f"{third.loc['pooled', 'change']:.0%}; 2023-2026 alone gives the same slope.",
    "Long replies typed while others keep posting get short gaps; that works against this result, not for it.",
]
fig.text(0.02, 0.01, "\n".join("• " + line for line in footnote), fontsize=8, color="dimgrey", va="bottom")
fig.tight_layout(rect=(0, 0.13, 1, 0.9))
fig.savefig("reply-length-final.png", dpi=120)